In [ ]:
# Instalar gdown para descargar el archivo desde Google Drive
!pip install -q gdown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    RepeatVector,
    TimeDistributed,
    Dropout,
    Input
)
from tensorflow.keras.callbacks import EarlyStopping


# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==========================================

# ID del archivo Dataset_Unificado_TOTAL.csv
FILE_ID = "1Xk2cwVHr17jWSZgP23b2ikUAKQHKYMs6"

# Ruta temporal dentro de Google Colab
ruta = "/content/Dataset_Unificado_TOTAL.csv"

# Descargar el archivo solamente si todavía no existe
if not os.path.exists(ruta):
    print("⬇️ Descargando Dataset_Unificado_TOTAL.csv...")

    archivo_descargado = gdown.download(
        id=FILE_ID,
        output=ruta,
        quiet=False
    )

    if archivo_descargado is None:
        raise RuntimeError(
            "❌ No se pudo descargar el archivo. "
            "Verifica que esté compartido como "
            "'Cualquier persona con el enlace → Lector'."
        )

print("📂 Cargando Dataset Unificado...")

df_total = pd.read_csv(
    ruta,
    encoding="latin-1",
    low_memory=False
)

# Limpiar espacios de los nombres de las columnas
df_total.columns = df_total.columns.str.strip()

print(f"Registros originales: {len(df_total):,}")
print("Columnas disponibles:")
print(df_total.columns.tolist())


# ==========================================
# 2. LIMITAR REGISTROS PARA LA PRUEBA
# ==========================================

# Solo se usarán 10 000 registros
LIMIT = 10000

if len(df_total) > LIMIT:
    df_total = df_total.iloc[:LIMIT].copy()

print(f"✅ Registros utilizados en la prueba: {len(df_total):,}")


# ==========================================
# 3. VERIFICAR Y LIMPIAR COLUMNAS
# ==========================================

columnas_necesarias = [
    "latitude",
    "longitude",
    "speed",
    "course"
]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_total.columns
]

if columnas_faltantes:
    raise ValueError(
        "❌ No se encontraron estas columnas: "
        f"{columnas_faltantes}\n"
        f"Columnas disponibles: {df_total.columns.tolist()}"
    )

# Convertir las columnas a valores numéricos
for columna in columnas_necesarias:
    df_total[columna] = pd.to_numeric(
        df_total[columna],
        errors="coerce"
    )

# Eliminar valores vacíos o infinitos
df_total = df_total.replace(
    [np.inf, -np.inf],
    np.nan
)

df_total = df_total.dropna(
    subset=columnas_necesarias
).reset_index(drop=True)

print(
    f"✅ Registros válidos después de la limpieza: "
    f"{len(df_total):,}"
)

if len(df_total) < 1000:
    raise ValueError(
        "❌ Quedaron muy pocos registros válidos para entrenar."
    )


# ==========================================
# 4. INGENIERÍA DE VARIABLES
# ==========================================

TIME_STEPS = 30

# Calcular cambios de posición
df_total["delta_lat"] = (
    df_total["latitude"]
    .diff()
    .fillna(0)
)

df_total["delta_lon"] = (
    df_total["longitude"]
    .diff()
    .fillna(0)
)

FEATURES = [
    "speed",
    "course",
    "delta_lat",
    "delta_lon"
]

df_total = df_total.dropna(
    subset=FEATURES
).reset_index(drop=True)


# ==========================================
# 5. DIVISIÓN DE DATOS
# ==========================================

# 80 % entrenamiento y 20 % prueba
train_size = int(len(df_total) * 0.80)

df_train = df_total.iloc[:train_size].copy()

df_test_raw = (
    df_total
    .iloc[train_size:]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Entrenamiento: {len(df_train):,} registros"
)

print(
    f"Prueba: {len(df_test_raw):,} registros"
)


# ==========================================
# 6. NORMALIZACIÓN
# ==========================================

scaler = MinMaxScaler()

scaler.fit(
    df_train[FEATURES]
)

X_train_scaled = scaler.transform(
    df_train[FEATURES]
)


# ==========================================
# 7. CREAR SECUENCIAS TEMPORALES
# ==========================================

def crear_secuencias(datos, time_steps):
    secuencias = []

    for i in range(len(datos) - time_steps):
        secuencias.append(
            datos[i:i + time_steps]
        )

    return np.asarray(
        secuencias,
        dtype=np.float32
    )


print("⏳ Generando secuencias temporales...")

X_train_seq = crear_secuencias(
    X_train_scaled,
    TIME_STEPS
)

print(
    f"Forma de las secuencias de entrenamiento: "
    f"{X_train_seq.shape}"
)


# ==========================================
# 8. CREAR EL LSTM AUTOENCODER
# ==========================================

model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    LSTM(
        32,
        activation="relu",
        return_sequences=False
    ),

    Dropout(0.2),

    RepeatVector(
        X_train_seq.shape[1]
    ),

    LSTM(
        32,
        activation="relu",
        return_sequences=True
    ),

    Dropout(0.2),

    TimeDistributed(
        Dense(
            X_train_seq.shape[2]
        )
    )
])

model.compile(
    optimizer="adam",
    loss="mse"
)

model.summary()


# ==========================================
# 9. ENTRENAR EL MODELO
# ==========================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

print("🚀 Iniciando entrenamiento de 3 épocas...")

history = model.fit(
    X_train_seq,
    X_train_seq,
    epochs=3,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    shuffle=False,
    verbose=1
)


# ==========================================
# 10. GRÁFICA DE ENTRENAMIENTO
# ==========================================

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Pérdida de entrenamiento"
)

plt.plot(
    history.history["val_loss"],
    label="Pérdida de validación"
)

plt.title(
    "Convergencia del modelo LSTM Autoencoder"
)

plt.xlabel("Época")
plt.ylabel("Error MSE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ==========================================
# 11. CALCULAR EL UMBRAL
# ==========================================

print("📏 Calculando el umbral de anomalía...")

X_train_pred = model.predict(
    X_train_seq,
    verbose=0
)

train_mae = np.mean(
    np.abs(
        X_train_pred - X_train_seq
    ),
    axis=(1, 2)
)

# Percentil 99.5 del error normal
UMBRAL = np.percentile(
    train_mae,
    99.5
)

print(
    f"✅ Umbral ajustado: {UMBRAL:.6f}"
)


# ==========================================
# 12. GRÁFICA DEL UMBRAL
# ==========================================

plt.figure(figsize=(10, 5))

sns.histplot(
    train_mae,
    bins=40,
    kde=True
)

plt.axvline(
    UMBRAL,
    linestyle="--",
    linewidth=2,
    label=f"Umbral = {UMBRAL:.4f}"
)

plt.title(
    "Distribución del error de reconstrucción"
)

plt.xlabel("Error MAE")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.show()


# ==========================================
# 13. SIMULAR ATAQUE EN LOS DATOS DE PRUEBA
# ==========================================

print("⚔️ Simulando ataque en el conjunto de prueba...")

df_test_attack = df_test_raw.copy()

# 0 representa normal y 1 representa ataque
df_test_attack["label"] = 0

# El ataque comienza en la mitad de los datos
indice_ataque = int(
    len(df_test_attack) * 0.5
)

# Alteración simulada de la latitud
df_test_attack.loc[
    indice_ataque:,
    "latitude"
] += 0.03

df_test_attack.loc[
    indice_ataque:,
    "label"
] = 1

# Recalcular los cambios de posición
df_test_attack["delta_lat"] = (
    df_test_attack["latitude"]
    .diff()
    .fillna(0)
)

df_test_attack["delta_lon"] = (
    df_test_attack["longitude"]
    .diff()
    .fillna(0)
)


# ==========================================
# 14. PREPARAR DATOS DE PRUEBA
# ==========================================

X_test_scaled = scaler.transform(
    df_test_attack[FEATURES]
)

X_test_seq = crear_secuencias(
    X_test_scaled,
    TIME_STEPS
)

test_pred = model.predict(
    X_test_seq,
    verbose=0
)

test_mae = np.mean(
    np.abs(
        test_pred - X_test_seq
    ),
    axis=(1, 2)
)

# Etiquetas reales ajustadas al número de secuencias
y_true = (
    df_test_attack["label"]
    .iloc[TIME_STEPS:]
    .values
)

# Clasificar como anomalía si supera el umbral
y_pred = (
    test_mae > UMBRAL
).astype(int)


# ==========================================
# 15. REPORTE DE CLASIFICACIÓN
# ==========================================

print("\n" + "=" * 45)
print("🏆 RESULTADO FINAL DE LA PRUEBA")
print("=" * 45)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Normal",
            "Ataque"
        ],
        zero_division=0
    )
)


# ==========================================
# 16. MATRIZ DE CONFUSIÓN
# ==========================================

matriz = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cbar=False,
    xticklabels=[
        "Normal",
        "Ataque"
    ],
    yticklabels=[
        "Normal",
        "Ataque"
    ]
)

plt.title(
    "Matriz de confusión final"
)

plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.tight_layout()
plt.show()


# ==========================================
# 17. GRÁFICA FINAL DE ANOMALÍAS
# ==========================================

plt.figure(figsize=(12, 6))

plt.plot(
    test_mae,
    label="Error de reconstrucción"
)

plt.axhline(
    UMBRAL,
    linestyle="--",
    linewidth=2,
    label=f"Umbral = {UMBRAL:.4f}"
)

# Ajustar la posición del ataque al tamaño de las secuencias
inicio_ataque_grafica = max(
    0,
    indice_ataque - TIME_STEPS
)

plt.axvspan(
    inicio_ataque_grafica,
    len(test_mae),
    alpha=0.2,
    label="Zona del ataque simulado"
)

plt.title(
    "Detección final de anomalías"
)

plt.xlabel("Secuencia temporal")
plt.ylabel("Error de reconstrucción")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    "\n✅ Prueba ejecutada correctamente "
    "con 10 000 registros y 3 épocas."
)

⬇️ Descargando Dataset_Unificado_TOTAL.csv...


Downloading...
From (original): https://drive.google.com/uc?id=1Xk2cwVHr17jWSZgP23b2ikUAKQHKYMs6
From (redirected): https://drive.google.com/uc?id=1Xk2cwVHr17jWSZgP23b2ikUAKQHKYMs6&confirm=t&uuid=a3617f28-3f64-4537-92a5-eeafb525661f
To: /content/Dataset_Unificado_TOTAL.csv
100%|██████████| 293M/293M [00:01<00:00, 249MB/s]


📂 Cargando Dataset Unificado...
Registros originales: 5,916,406
Columnas disponibles:
['latitude', 'longitude', 'speed', 'course', 'Source']
✅ Registros utilizados en la prueba: 10,000
✅ Registros válidos después de la limpieza: 10,000
Entrenamiento: 8,000 registros
Prueba: 2,000 registros
⏳ Generando secuencias temporales...
Forma de las secuencias de entrenamiento: (7970, 30, 4)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 30, 4)          │           132 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,188 (51.52 KB)

 Trainable params: 13,188 (51.52 KB)

 Non-trainable params: 0 (0.00 B)

🚀 Iniciando entrenamiento de 3 épocas...
Epoch 1/3
 24/113 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.2779